# Project Bài 9 · Tìm sản phẩm trông giống với ít con số

**Học máy cơ bản · Bài 9 · PGS.TS Nguyễn Văn Hậu**

**Tình huống.** Một cửa hàng thời trang trực tuyến muốn thêm nút “sản phẩm trông giống”: khi khách xem một ảnh, hệ thống tìm ảnh gần nhất trong kho. Kho lưu được tối đa **10% số con số** so với lưu nguyên 784 pixel mỗi ảnh. Bạn dùng PCA để giảm chiều, rồi tìm láng giềng gần nhất (1-NN) trên score. Đo chất lượng bằng tỷ lệ ảnh tìm được **cùng loại** với ảnh truy vấn.

**Dữ liệu.** `data/project_subset.npz`: 10.000 ảnh Fashion-MNIST (Zalando Research, giấy phép MIT) lấy từ 45.000 ảnh train chính thức **không dùng trong bài giảng** (seed 2026): 8.000 ảnh kho (fit), 1.000 ảnh validation, 1.000 ảnh kiểm tra cuối. Mỗi loại đúng một phần mười. Test chính thức vẫn để dành cho Bài 10.

| Mốc | Bước | Ô tự kiểm |
|---|---|---|
| M1 | 1: nạp, kiểm số ảnh | 8.000 / 1.000 / 1.000; mỗi loại 10% |
| M2 | 2: tự cài PCA, đối chiếu scikit-learn | trị riêng, abs(cos), dấu tuỳ ý |
| M3 | 3: EVR, k90, lỗi tái tạo trên validation | k90; MSE giảm theo k |
| M4 | 4 (khá): 1-NN theo lưới k, ràng buộc lưu trữ, chọn k | tỷ lệ lưu trữ; k lớn nhất được phép |
| M5 | 5 (khá): chấm kiểm tra cuối **một lần** | hàm chặn chấm lần hai |

**Ba mức.** *Cơ bản (5 điểm)*: bước 1–3, chọn một k thoả ràng buộc và báo 1-NN trên validation. *Khá (+3)*: bước 4–5 và báo cáo một trang. *Nâng cao (+2)*: A (lặp lũy thừa có khử cho 5 PC đầu) hoặc B (ảnh truy vấn bị dịch).

**Luật chơi.** Bước 2 không dùng `PCA` của thư viện để tính; chỉ dùng để đối chiếu ở ô có ghi rõ. Mọi lựa chọn (k, quy tắc chọn) làm trên validation. Tập kiểm tra cuối chỉ chấm một lần, bằng hàm `cham_cuoi` (đã viết sẵn, không sửa).

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import json, time
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path.cwd()
assert (ROOT / 'data' / 'project_subset.npz').exists(), 'Mở notebook từ thư mục bài (có thư mục data/).'
with np.load(ROOT / 'data' / 'project_subset.npz') as d:
    X = d['fit_images'] / 255.0; y = d['fit_labels']
    V = d['val_images'] / 255.0; yv = d['val_labels']
    _F, _yF = d['final_images'] / 255.0, d['final_labels']      # chỉ dùng qua cham_cuoi

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

_DAU_VET = ROOT / '.project_bai09_cham_cuoi.json'
def cham_cuoi(k, mu, Wk):
    # Chấm tập kiểm tra cuối đúng MỘT lần. Lần sau chỉ được gọi lại với đúng k đã chấm (để chạy lại notebook); đổi k sẽ bị chặn.
    if _DAU_VET.exists():                          # kiểm khoá trước khi tính bất cứ gì trên tập cuối
        cu = json.loads(_DAU_VET.read_text())
        if cu['k'] != k:
            raise RuntimeError(f"Tập kiểm tra cuối đã được chấm với k = {cu['k']} lúc {cu['luc']}. Không được chấm lại với k khác.")
        return cu['acc']
    Zk = (X - mu) @ Wk; Zq = (_F - mu) @ Wk
    acc = float(KNeighborsClassifier(1).fit(Zk, y).score(Zq, _yF))
    _DAU_VET.write_text(json.dumps(dict(k=k, acc=acc, luc=time.strftime('%Y-%m-%d %H:%M'))))
    return acc
print(X.shape, V.shape, _F.shape)

## Bước 1 · Kiểm dữ liệu

Đếm số ảnh mỗi tập và mỗi loại (10 loại, nhãn 0…9).

In [ ]:
# TODO: dem = {'fit': số ảnh mỗi loại của y, 'val': số ảnh mỗi loại của yv}  (gợi ý: np.bincount)
raise NotImplementedError('TODO bước 1: dem')
print(dem)

In [ ]:
# TỰ KIỂM · bước 1
kiem('8.000 ảnh kho, 1.000 ảnh validation', (len(X), len(V)) == (8000, 1000))
kiem('mỗi loại đúng 10%', all((v == v[0]).all() for v in dem.values()))

## Bước 2 · Tự cài PCA

Viết `pca_fit(X)` trả `mu`, các trị riêng **giảm dần** và ma trận vector riêng tương ứng (cột j là thành phần j). Dùng ma trận hiệp phương sai chia n − 1 và `np.linalg.eigh`. Viết `chieu(M, mu, Wk)` và `tai_tao(Z, mu, Wk)`.

In [ ]:
def pca_fit(X):
    # TODO: mu, C (chia n − 1), eigh, sắp xếp giảm dần; trả mu, lam, W
    raise NotImplementedError('TODO bước 2: pca_fit')
def chieu(M, mu, Wk):
    raise NotImplementedError('TODO bước 2: chieu')
def tai_tao(Z, mu, Wk):
    raise NotImplementedError('TODO bước 2: tai_tao')
mu, lam, W = pca_fit(X)
print('λ1, λ2 =', lam[:2].round(4))

In [ ]:
# TỰ KIỂM · bước 2 (scikit-learn chỉ dùng để đối chiếu)
sk = PCA().fit(X)
tot = lam > 1e-6 * lam[0]
kiem('trị riêng khớp scikit-learn', np.allclose(lam[tot], sk.explained_variance_[tot], rtol=1e-8), 'chia n − 1? đã sắp giảm dần chưa?')
cos = np.abs(np.sum(W[:, :50] * sk.components_[:50].T, 0))
kiem('50 thành phần đầu trùng hướng (|cos| ≈ 1)', cos.min() > 1 - 1e-8, 'so |cos|, vì dấu của vector riêng là tuỳ ý')
kiem('λ1 = 19.4842', abs(lam[0] - 19.484224397975186) < 1e-8)
print('dấu 5 PC đầu so với thư viện:', np.sign(np.sum(W[:, :5] * sk.components_[:5].T, 0)))

## Bước 3 · EVR, k90 và lỗi tái tạo trên validation

`evr[k-1]` là EVR cộng dồn của k thành phần đầu (trên tập kho). Viết `mse_val(k)`: MSE mỗi pixel của validation khi giữ k thành phần (k = 0 nghĩa là chỉ dùng μ).

In [ ]:
LUOI = [8, 16, 32, 48, 64, 82, 128]
# TODO: evr (cộng dồn), k90 (k nhỏ nhất có evr ≥ 0,90), hàm mse_val(k)
raise NotImplementedError('TODO bước 3: evr, k90, mse_val')
bang_mse = {k: mse_val(k) for k in [0] + LUOI}
print('k90 =', k90, '· EVR k = 64:', round(evr[63], 4)); print({k: round(v, 4) for k, v in bang_mse.items()})

In [ ]:
# TỰ KIỂM · bước 3
kiem('k90 = 83', k90 == 83)
kiem('MSE k = 0 và k = 64 đúng', abs(bang_mse[0] - 0.08562666598623708) < 1e-9 and abs(bang_mse[64] - 0.01023996564916711) < 1e-9, 'k = 0: tái tạo chỉ bằng μ')
kiem('MSE giảm khi k tăng', all(np.diff(list(bang_mse.values())) < 0))

## Bước 4 (khá) · 1-NN theo lưới k, ràng buộc lưu trữ, chọn k

Số con số phải lưu cho kho: score (n × k) + các hướng (784 × k) + μ (784). Viết `ty_le_luu(k)` so với n × 784. Đo accuracy 1-NN trên validation cho mỗi k trong lưới.

**Quy tắc chọn (ghi trước khi xem kết quả):** trong các k thoả ràng buộc ≤ 10%, lấy accuracy cao nhất `a*`; chọn **k nhỏ nhất** có accuracy ≥ `a*` − một sai số chuẩn của `a*` (sai số chuẩn = √(a(1 − a)/1.000)).

In [ ]:
# TODO: ty_le_luu(k); acc[k] = accuracy 1-NN trên validation dùng k score đầu; hop_le; a_sao; se_sao; k_chon theo quy tắc
raise NotImplementedError('TODO bước 4')
for k in LUOI:
    print(f'k = {k:3d}: lưu {ty_le_luu(k):6.1%} · 1-NN validation {acc[k]:.3f}' + ('' if ty_le_luu(k) <= 0.10 else '  (vượt 10%)'))
print('k chọn =', k_chon)

In [ ]:
# TỰ KIỂM · bước 4
kiem('k lớn nhất được phép = 71', ty_le_luu(71) <= 0.10 < ty_le_luu(72))
kiem('k = 82 vượt ràng buộc, k = 64 thì không', ty_le_luu(82) > 0.10 >= ty_le_luu(64))
kiem('accuracy k = 64 đúng', abs(acc[64] - 0.81) < 1e-12)
kiem('k chọn đúng quy tắc (nhỏ nhất trong một sai số chuẩn của mức tốt nhất, trong các k được phép)',
     k_chon in hop_le and k_chon == min(k for k in hop_le if acc[k] >= max(acc[j] for j in hop_le) - se_sao))

## Bước 5 (khá) · Chấm tập kiểm tra cuối một lần

Chỉ chạy ô này **sau khi** đã chốt `k_chon`. Hàm `cham_cuoi` ghi lại lần chấm vào `.project_bai09_cham_cuoi.json`. Chạy lại notebook (Restart & Run All) với **cùng** k thì nhận lại đúng kết quả cũ; đổi k sau khi đã chấm sẽ bị chặn — đó là chủ đích: tập kiểm tra cuối không được dùng để chọn k. Ô tự kiểm không cho biết đáp số của tập này.

In [ ]:
acc_cuoi = cham_cuoi(k_chon, mu, W[:, :k_chon])
se_cuoi = np.sqrt(acc_cuoi * (1 - acc_cuoi) / 1000)
print(f'1-NN trên tập kiểm tra cuối, k = {k_chon}: {acc_cuoi:.3f} ± {se_cuoi:.3f} (một sai số chuẩn)')

In [ ]:
# TỰ KIỂM · bước 5
kiem('kết quả chấm cuối là một tỷ lệ hợp lệ', 0 <= acc_cuoi <= 1)
try:
    cham_cuoi(k_chon + 1, mu, W[:, :k_chon + 1]); chan = False
except RuntimeError:
    chan = True
kiem('hàm chặn chấm lần hai với k khác', chan)

## Báo cáo một trang (nộp cùng notebook)

1. **Dữ liệu.** Ba tập, cách lấy, vì sao không dùng test chính thức.
2. **PCA.** Bằng chứng mã tự viết trùng thư viện (trị riêng, |cos|, dấu tuỳ ý); EVR và k90.
3. **Chọn k.** Bảng lưu trữ – lỗi tái tạo – 1-NN theo k; quy tắc chọn đã ghi trước; k được chọn và lý do (không chọn theo “EVR 90%”).
4. **Kết quả cuối.** Accuracy trên tập kiểm tra cuối kèm sai số chuẩn; so với validation.
5. **Giới hạn.** Ít nhất ba: ảnh đã được Zalando căn giữa (ảnh khách chụp thì không); một lần chia, một seed; “cùng loại” chưa phải “trông giống” theo cảm nhận của khách; số con số chưa phải số byte.

## Nâng cao (chọn một)

- **A · Lặp lũy thừa có khử.** Viết `lap_luy_thua(C, so_pc=5)`: với mỗi thành phần, bắt đầu từ vector đều, lặp `w ← Cw/‖Cw‖` tới khi `1 − |w_mới · w_cũ| < 1e-13`, lấy λ = wᵀCw, rồi khử `C ← C − λwwᵀ`. So góc với `W[:, j]` và λ với `lam[j]`. Giải thích vì sao PC5 cần nhiều vòng hơn PC3 (gợi ý: tỉ số λ₆/λ₅ ≈ 0,89 so với λ₄/λ₃ ≈ 0,83).
- **B · Ảnh truy vấn bị dịch.** Dịch ảnh validation sang phải 1 và 2 pixel (ảnh kho giữ nguyên), đo lại 1-NN với `k_chon`. Accuracy giảm bao nhiêu? Đề xuất và thử một cách giảm (ví dụ căn khung bao của món đồ về giữa ảnh trước khi chiếu), đo trên validation.

In [ ]:
# Nâng cao A · lời giải tham khảo (sinh viên tự viết)
# TODO (tuỳ chọn): viết lap_luy_thua(C, so_pc=5) theo mô tả ở trên

In [ ]:
# Nâng cao B · lời giải tham khảo (sinh viên tự làm)
# TODO (tuỳ chọn): dịch ảnh validation 1 và 2 pixel, đo lại 1-NN với k_chon